<a href="https://colab.research.google.com/github/gustavopatinotoro/CienciaDeDatos_SECOPII/blob/main/notebooks/BOOTSTRAP_02_C_%E2%80%94_Recuperaci%C3%B3n_certificada_del_laboratorio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =============================================================================
# SECOP-RT LAB
# NOTEBOOK 02 — AUDITORÍA SEMÁNTICA DENSA
#
# BOOTSTRAP:
#   BOOTSTRAP-02-C — Recuperación certificada desde G1.11-C
#
# VERSIÓN:
#   1.0.0
#
# PROPÓSITO:
#
#   Reconstruir un entorno de trabajo completamente nuevo utilizando
#   exclusivamente artefactos persistentes certificados en Google Drive.
#
# PUNTO DE RECUPERACIÓN:
#
#   Gate:
#       G1.11-C
#
#   SHA-256 manifiesto esperado:
#       9815f09a982d4caf299dfe7a9cca0f12d0173efa6db6c149cb44c04219a70e8e
#
# PRINCIPIOS:
#
#   - No depende de variables del notebook anterior.
#   - No modifica datos RAW.
#   - No modifica Parquet persistente.
#   - No modifica estado_laboratorio.json.
#   - No instala paquetes.
#   - No ejecuta G0/G1 nuevamente.
#   - Verifica la cadena de manifiestos hasta G0.5-C.
#   - Verifica y copia una sola vez las 188 partes Parquet a /content.
#   - Verifica la referencia UNSPSC congelada.
#   - Verifica los artefactos G1.10-C y G1.11-C.
#   - Produce un estado LOCAL de sesión para G1.12+.
#
# RECUPERACIÓN:
#
#   Si Colab reinicia el runtime:
#
#       volver a ejecutar únicamente BOOTSTRAP-02-C.
#
#   Después pueden ejecutarse G1.12-C y gates posteriores.
#
# ESTADO:
#
#   PASS:
#       entorno local reconstruido y certificado.
#
#   FAIL:
#       ruptura de integridad de un artefacto persistido.
#
#   BLOCKED:
#       recurso requerido no disponible.
# =============================================================================


from __future__ import annotations

import hashlib
import importlib.metadata
import json
import os
import platform
import shutil
import sys

from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as pads

from google.colab import drive


# =============================================================================
# 1. CONTRATO DE RECUPERACIÓN
# =============================================================================

BOOTSTRAP = "BOOTSTRAP-02-C"
VERSION_BOOTSTRAP = "1.0.0"

GATE_BASE_ESPERADO = "G1.11-C"

SHA_MANIFIESTO_BASE_ESPERADO = (
    "9815f09a982d4caf299dfe7a9cca0f12"
    "d0173efa6db6c149cb44c04219a70e8e"
)

SNAPSHOT_ESPERADO = (
    "secop2_jbjy-vk9h_ru1792151761_n5633184_"
    "20261016T181714_602776Z"
)

FILAS_ESPERADAS = 5_633_184
COLUMNAS_ESPERADAS = 98
PARTES_ESPERADAS = 188


# =============================================================================
# 2. DRIVE
# =============================================================================

RUTA_DRIVE = Path(
    "/content/drive"
)

RUTA_MYDRIVE = (
    RUTA_DRIVE
    / "MyDrive"
)


if not RUTA_MYDRIVE.is_dir():

    drive.mount(
        str(
            RUTA_DRIVE
        ),
        force_remount=False,
    )


if not RUTA_MYDRIVE.is_dir():

    raise RuntimeError(
        "BOOTSTRAP-02-C BLOCKED: Google Drive no disponible."
    )


# =============================================================================
# 3. RUTAS PERSISTENTES
# =============================================================================

RUTA_RAIZ = (
    RUTA_MYDRIVE
    / "SECOP_RT_Lab"
)

RUTA_ARTEFACTOS = (
    RUTA_RAIZ
    / "artefactos"
)

RUTA_CHECKPOINTS = (
    RUTA_RAIZ
    / "checkpoints"
)

RUTA_LOGS = (
    RUTA_RAIZ
    / "logs"
)

RUTA_ESTADO_GLOBAL = (
    RUTA_CHECKPOINTS
    / "estado_laboratorio.json"
)


if not RUTA_ESTADO_GLOBAL.exists():

    raise RuntimeError(
        "BOOTSTRAP-02-C FAIL: checkpoint global ausente."
    )


RUTA_LOGS.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# 4. WORKSPACE LOCAL EFÍMERO
# =============================================================================

RUTA_LOCAL = Path(
    "/content/secop_rt_notebook02"
)

RUTA_LOCAL_SNAPSHOT = (
    RUTA_LOCAL
    / "snapshot"
)

RUTA_LOCAL_PARQUET = (
    RUTA_LOCAL_SNAPSHOT
    / "parquet"
)

RUTA_LOCAL_REFERENCIA = (
    RUTA_LOCAL
    / "referencia_unspsc"
)

RUTA_LOCAL_G110 = (
    RUTA_LOCAL
    / "g1_10"
)

RUTA_LOCAL_G111 = (
    RUTA_LOCAL
    / "g1_11"
)

RUTA_LOCAL_ESTADO = (
    RUTA_LOCAL
    / "bootstrap_02_estado.json"
)


for ruta in (
    RUTA_LOCAL,
    RUTA_LOCAL_SNAPSHOT,
    RUTA_LOCAL_PARQUET,
    RUTA_LOCAL_REFERENCIA,
    RUTA_LOCAL_G110,
    RUTA_LOCAL_G111,
):

    ruta.mkdir(
        parents=True,
        exist_ok=True,
    )


# =============================================================================
# 5. UTILIDADES DE INTEGRIDAD
# =============================================================================

def sha256_archivo(
    ruta: Path,
    bloque: int = 4 * 1024 * 1024,
) -> str:

    digest = hashlib.sha256()

    with ruta.open(
        "rb"
    ) as archivo:

        while True:

            datos = archivo.read(
                bloque
            )

            if not datos:
                break

            digest.update(
                datos
            )

    return digest.hexdigest()


def escribir_json_atomico(
    ruta: Path,
    objeto: Any,
) -> None:

    temporal = ruta.with_name(
        "."
        + ruta.name
        + ".tmp"
    )

    temporal.write_text(
        json.dumps(
            objeto,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
            default=str,
        )
        + "\n",
        encoding="utf-8",
    )

    json.loads(
        temporal.read_text(
            encoding="utf-8"
        )
    )

    os.replace(
        temporal,
        ruta,
    )


def cargar_json_verificado(
    ruta: Path,
    sha_esperado: str,
    etiqueta: str,
) -> dict[str, Any]:

    if not ruta.exists():

        raise RuntimeError(
            f"BOOTSTRAP-02-C FAIL: {etiqueta} ausente: {ruta}"
        )


    sha_observado = sha256_archivo(
        ruta
    )


    if sha_observado != sha_esperado:

        raise RuntimeError(
            f"BOOTSTRAP-02-C FAIL: SHA-256 divergente en {etiqueta}."
        )


    return json.loads(
        ruta.read_text(
            encoding="utf-8"
        )
    )


def verificar_archivo(
    ruta: Path,
    sha_esperado: str,
    etiqueta: str,
) -> None:

    if not ruta.exists():

        raise RuntimeError(
            f"BOOTSTRAP-02-C FAIL: {etiqueta} ausente: {ruta}"
        )


    sha = sha256_archivo(
        ruta
    )


    if sha != sha_esperado:

        raise RuntimeError(
            f"BOOTSTRAP-02-C FAIL: SHA-256 divergente en {etiqueta}."
        )


def copiar_verificado(
    origen: Path,
    destino: Path,
    sha_esperado: str,
    etiqueta: str,
) -> None:

    destino.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    if destino.exists():

        if sha256_archivo(
            destino
        ) == sha_esperado:

            return


        destino.unlink()


    if not origen.exists():

        raise RuntimeError(
            f"BOOTSTRAP-02-C FAIL: origen ausente para {etiqueta}: "
            f"{origen}"
        )


    shutil.copyfile(
        origen,
        destino,
    )


    sha_destino = sha256_archivo(
        destino
    )


    if sha_destino != sha_esperado:

        destino.unlink(
            missing_ok=True
        )

        raise RuntimeError(
            f"BOOTSTRAP-02-C FAIL: copia local corrupta en {etiqueta}."
        )


def verificar_artefactos(
    manifiesto: dict[str, Any],
    etiqueta: str,
) -> dict[str, Path]:

    resultado = {}


    artefactos = manifiesto.get(
        "artefactos",
        {},
    )


    for nombre, info in artefactos.items():

        if not isinstance(
            info,
            dict,
        ):

            continue


        ruta_texto = info.get(
            "ruta"
        )

        sha = info.get(
            "sha256"
        )


        if (
            ruta_texto is None
            or
            sha is None
        ):

            continue


        ruta = Path(
            ruta_texto
        )


        verificar_archivo(
            ruta,
            str(
                sha
            ),
            f"{etiqueta}/{nombre}",
        )


        resultado[
            nombre
        ] = ruta


    return resultado


# =============================================================================
# 6. PROTEGER CHECKPOINT GLOBAL
# =============================================================================

SHA_ESTADO_GLOBAL_ANTES = sha256_archivo(
    RUTA_ESTADO_GLOBAL
)


estado = json.loads(
    RUTA_ESTADO_GLOBAL.read_text(
        encoding="utf-8"
    )
)


if (
    estado.get(
        "ultimo_gate_cerrado"
    )
    != GATE_BASE_ESPERADO
):

    raise RuntimeError(
        "BOOTSTRAP-02-C FAIL: punto de recuperación inesperado. "
        f"Observado={estado.get('ultimo_gate_cerrado')!r}; "
        f"esperado={GATE_BASE_ESPERADO!r}."
    )


if (
    estado.get(
        "sha256_ultimo_manifiesto"
    )
    != SHA_MANIFIESTO_BASE_ESPERADO
):

    raise RuntimeError(
        "BOOTSTRAP-02-C FAIL: el checkpoint no apunta al "
        "manifiesto G1.11-C certificado esperado."
    )


# =============================================================================
# 7. G1.11-C
# =============================================================================

ruta_g111 = Path(
    estado[
        "ultimo_manifiesto"
    ]
)


g111 = cargar_json_verificado(
    ruta_g111,
    SHA_MANIFIESTO_BASE_ESPERADO,
    "manifiesto G1.11-C",
)


if (
    g111[
        "proyecto"
    ][
        "gate"
    ]
    != "G1.11-C"
):

    raise RuntimeError(
        "BOOTSTRAP-02-C FAIL: gate de manifiesto base inesperado."
    )


SNAPSHOT_ID = (
    g111[
        "fuente"
    ][
        "snapshot_id"
    ]
)


FINGERPRINT_PARQUET = (
    g111[
        "fuente"
    ][
        "fingerprint_parquet"
    ]
)


if SNAPSHOT_ID != SNAPSHOT_ESPERADO:

    raise RuntimeError(
        "BOOTSTRAP-02-C FAIL: snapshot inesperado."
    )


# =============================================================================
# 8. G1.10-C
# =============================================================================

ruta_g110 = Path(
    g111[
        "fuente"
    ][
        "manifiesto_g110c"
    ]
)

sha_g110 = str(
    g111[
        "fuente"
    ][
        "sha256_manifiesto_g110c"
    ]
)


g110 = cargar_json_verificado(
    ruta_g110,
    sha_g110,
    "manifiesto G1.10-C",
)


# =============================================================================
# 9. G1.9-C
# =============================================================================

ruta_g19 = Path(
    g110[
        "fuente"
    ][
        "manifiesto_g19c"
    ]
)

sha_g19 = str(
    g110[
        "fuente"
    ][
        "sha256_manifiesto_g19c"
    ]
)


g19 = cargar_json_verificado(
    ruta_g19,
    sha_g19,
    "manifiesto G1.9-C",
)


# =============================================================================
# 10. G1.8-C
# =============================================================================

ruta_g18 = Path(
    g19[
        "fuente"
    ][
        "manifiesto_g18c"
    ]
)

sha_g18 = str(
    g19[
        "fuente"
    ][
        "sha256_manifiesto_g18c"
    ]
)


g18 = cargar_json_verificado(
    ruta_g18,
    sha_g18,
    "manifiesto G1.8-C",
)


# =============================================================================
# 11. G1.7-C
#
# G1.8-C utiliza "fuente_secop".
# =============================================================================

ruta_g17 = Path(
    g18[
        "fuente_secop"
    ][
        "manifiesto_g17c"
    ]
)

sha_g17 = str(
    g18[
        "fuente_secop"
    ][
        "sha256_manifiesto_g17c"
    ]
)


g17 = cargar_json_verificado(
    ruta_g17,
    sha_g17,
    "manifiesto G1.7-C",
)


# =============================================================================
# 12. G1.6 → G0.5
# =============================================================================

def cargar_desde_fuente(
    padre: dict[str, Any],
    clave_ruta: str,
    clave_sha: str,
    etiqueta: str,
) -> tuple[
    Path,
    dict[str, Any],
    str,
]:

    fuente = padre[
        "fuente"
    ]


    ruta = Path(
        fuente[
            clave_ruta
        ]
    )


    sha = str(
        fuente[
            clave_sha
        ]
    )


    objeto = cargar_json_verificado(
        ruta,
        sha,
        etiqueta,
    )


    return (
        ruta,
        objeto,
        sha,
    )


ruta_g16, g16, sha_g16 = cargar_desde_fuente(
    g17,
    "manifiesto_g16c",
    "sha256_manifiesto_g16c",
    "G1.6-C",
)


ruta_g15, g15, sha_g15 = cargar_desde_fuente(
    g16,
    "manifiesto_g15c",
    "sha256_manifiesto_g15c",
    "G1.5-C",
)


ruta_g14, g14, sha_g14 = cargar_desde_fuente(
    g15,
    "manifiesto_g14c",
    "sha256_manifiesto_g14c",
    "G1.4-C",
)


ruta_g13, g13, sha_g13 = cargar_desde_fuente(
    g14,
    "manifiesto_g13c",
    "sha256_manifiesto_g13c",
    "G1.3-C",
)


ruta_g12, g12, sha_g12 = cargar_desde_fuente(
    g13,
    "manifiesto_g12c",
    "sha256_manifiesto_g12c",
    "G1.2-C",
)


ruta_g11, g11, sha_g11 = cargar_desde_fuente(
    g12,
    "manifiesto_g11c",
    "sha256_manifiesto_g11c",
    "G1.1-C",
)


ruta_g10, g10, sha_g10 = cargar_desde_fuente(
    g11,
    "manifiesto_g10c",
    "sha256_manifiesto_g10c",
    "G1.0-C",
)


ruta_g05, g05, sha_g05 = cargar_desde_fuente(
    g10,
    "manifiesto_g05c",
    "sha256_manifiesto_g05c",
    "G0.5-C",
)


# =============================================================================
# 13. CONSISTENCIA DEL SNAPSHOT
# =============================================================================

if (
    g05[
        "parquet"
    ][
        "fingerprint_sha256"
    ]
    != FINGERPRINT_PARQUET
):

    raise RuntimeError(
        "BOOTSTRAP-02-C FAIL: fingerprint G0.5 != G1.11."
    )


# =============================================================================
# 14. CHECKPOINT G0.5
# =============================================================================

ruta_checkpoint_g05 = Path(
    g05[
        "checkpoint"
    ][
        "ruta"
    ]
)


sha_checkpoint_g05 = str(
    g05[
        "checkpoint"
    ][
        "sha256"
    ]
)


checkpoint_g05 = cargar_json_verificado(
    ruta_checkpoint_g05,
    sha_checkpoint_g05,
    "checkpoint G0.5-C",
)


PARTES = checkpoint_g05[
    "partes"
]


if len(
    PARTES
) != PARTES_ESPERADAS:

    raise RuntimeError(
        "BOOTSTRAP-02-C FAIL: número de partes inesperado."
    )


# =============================================================================
# 15. VERIFICAR ARTEFACTOS G1.10 Y G1.11
# =============================================================================

artefactos_g110 = verificar_artefactos(
    g110,
    "G1.10-C",
)


artefactos_g111 = verificar_artefactos(
    g111,
    "G1.11-C",
)


# =============================================================================
# 16. VERIFICAR REFERENCIA UNSPSC
# =============================================================================

ruta_zip_unspsc = Path(
    g111[
        "referencia_unspsc"
    ][
        "zip"
    ]
)


sha_zip_unspsc = str(
    g111[
        "referencia_unspsc"
    ][
        "sha256_zip"
    ]
)


verificar_archivo(
    ruta_zip_unspsc,
    sha_zip_unspsc,
    "ZIP UNSPSC certificado",
)


ruta_catalogo_unique = Path(
    g110[
        "referencia_unspsc"
    ][
        "ruta_catalogo_unique"
    ]
)


sha_catalogo_unique = str(
    g110[
        "referencia_unspsc"
    ][
        "sha256_catalogo_unique"
    ]
)


verificar_archivo(
    ruta_catalogo_unique,
    sha_catalogo_unique,
    "catálogo UNSPSC UNIQUE",
)


# =============================================================================
# 17. CAPACIDAD LOCAL
# =============================================================================

bytes_parquet = sum(
    Path(
        parte[
            "ruta"
        ]
    ).stat().st_size
    for parte in PARTES
)


espacio_libre = shutil.disk_usage(
    "/content"
).free


margen_requerido = (
    bytes_parquet
    + 1_500_000_000
)


if espacio_libre < margen_requerido:

    raise RuntimeError(
        "BOOTSTRAP-02-C BLOCKED: espacio local insuficiente. "
        f"Necesario≈{margen_requerido / 2**30:.2f} GiB, "
        f"libre={espacio_libre / 2**30:.2f} GiB."
    )


# =============================================================================
# 18. STAGE LOCAL PARQUET — UNA SOLA VEZ POR RUNTIME
# =============================================================================

print("=" * 108)
print("SECOP-RT LAB")
print("BOOTSTRAP-02-C — RECUPERACIÓN CERTIFICADA")
print("=" * 108)

print(
    f"Gate base:                       {GATE_BASE_ESPERADO}"
)

print(
    f"Snapshot:                        {SNAPSHOT_ID}"
)

print(
    f"Partes Parquet:                  {len(PARTES)}"
)

print(
    f"Tamaño Parquet aprox.:           {bytes_parquet / 2**30:.3f} GiB"
)

print()
print(
    "Reconstruyendo stage local..."
)


n_reutilizadas = 0
n_copiadas = 0


for posicion, parte in enumerate(
    PARTES,
    start=1,
):

    origen = Path(
        parte[
            "ruta"
        ]
    )


    sha_esperado = str(
        parte[
            "sha256_parquet"
        ]
    )


    destino = (
        RUTA_LOCAL_PARQUET
        / origen.name
    )


    ya_valida = (
        destino.exists()
        and
        sha256_archivo(
            destino
        )
        == sha_esperado
    )


    if ya_valida:

        n_reutilizadas += 1


    else:

        copiar_verificado(
            origen,
            destino,
            sha_esperado,
            f"Parquet {posicion}",
        )

        n_copiadas += 1


    if (
        posicion % 25 == 0
        or
        posicion == len(
            PARTES
        )
    ):

        print(
            f"  stage+SHA: {posicion}/{len(PARTES)}"
        )


# =============================================================================
# 19. ASEGURAR QUE NO HAY PARTES EXTRA
# =============================================================================

nombres_esperados = {
    Path(
        parte[
            "ruta"
        ]
    ).name
    for parte in PARTES
}


for ruta_extra in RUTA_LOCAL_PARQUET.glob(
    "*.parquet"
):

    if ruta_extra.name not in nombres_esperados:

        ruta_extra.unlink()


if len(
    list(
        RUTA_LOCAL_PARQUET.glob(
            "*.parquet"
        )
    )
) != PARTES_ESPERADAS:

    raise RuntimeError(
        "BOOTSTRAP-02-C FAIL: stage local no contiene exactamente "
        "188 partes."
    )


# =============================================================================
# 20. CERTIFICAR DATASET LOCAL
# =============================================================================

dataset = pads.dataset(
    str(
        RUTA_LOCAL_PARQUET
    ),
    format="parquet",
)


FILAS_ARROW = int(
    dataset.count_rows()
)


SCHEMA_LOCAL = dataset.schema


if FILAS_ARROW != FILAS_ESPERADAS:

    raise RuntimeError(
        "BOOTSTRAP-02-C FAIL: filas Arrow locales divergentes."
    )


if len(
    SCHEMA_LOCAL
) != COLUMNAS_ESPERADAS:

    raise RuntimeError(
        "BOOTSTRAP-02-C FAIL: schema local != 98 columnas."
    )


if not all(
    pa.types.is_string(
        campo.type
    )
    for campo in SCHEMA_LOCAL
):

    raise RuntimeError(
        "BOOTSTRAP-02-C FAIL: Parquet local dejó de ser lexical STRING."
    )


# =============================================================================
# 21. VALIDACIÓN DUCKDB
# =============================================================================

glob_sql = str(
    RUTA_LOCAL_PARQUET
    / "*.parquet"
).replace(
    "'",
    "''",
)


con = duckdb.connect(
    database=":memory:"
)


FILAS_DUCKDB = int(
    con.execute(
        f"""
        SELECT COUNT(*)
        FROM read_parquet('{glob_sql}')
        """
    ).fetchone()[0]
)


con.close()


if FILAS_DUCKDB != FILAS_ESPERADAS:

    raise RuntimeError(
        "BOOTSTRAP-02-C FAIL: filas DuckDB locales divergentes."
    )


# =============================================================================
# 22. STAGE REFERENCIA UNSPSC
# =============================================================================

ruta_zip_local = (
    RUTA_LOCAL_REFERENCIA
    / "catalogo_unspsc.zip"
)


copiar_verificado(
    ruta_zip_unspsc,
    ruta_zip_local,
    sha_zip_unspsc,
    "ZIP UNSPSC local",
)


ruta_catalogo_local = (
    RUTA_LOCAL_REFERENCIA
    / "catalogo_nueva_version_core8_unique.csv"
)


copiar_verificado(
    ruta_catalogo_unique,
    ruta_catalogo_local,
    sha_catalogo_unique,
    "catálogo UNSPSC UNIQUE local",
)


# =============================================================================
# 23. STAGE ARTEFACTOS NECESARIOS G1.10-C
# =============================================================================

artefactos_locales_g110 = {}


for nombre in (
    "match",
    "metadata_referencia",
    "evidencia_jerarquia",
):

    if nombre not in artefactos_g110:

        raise RuntimeError(
            f"BOOTSTRAP-02-C FAIL: falta artefacto G1.10/{nombre}."
        )


    origen = artefactos_g110[
        nombre
    ]


    sha = str(
        g110[
            "artefactos"
        ][
            nombre
        ][
            "sha256"
        ]
    )


    destino = (
        RUTA_LOCAL_G110
        / origen.name
    )


    copiar_verificado(
        origen,
        destino,
        sha,
        f"G1.10/{nombre}",
    )


    artefactos_locales_g110[
        nombre
    ] = str(
        destino
    )


# =============================================================================
# 24. STAGE ARTEFACTOS G1.11-C
# =============================================================================

artefactos_locales_g111 = {}


for nombre, origen in artefactos_g111.items():

    sha = str(
        g111[
            "artefactos"
        ][
            nombre
        ][
            "sha256"
        ]
    )


    destino = (
        RUTA_LOCAL_G111
        / origen.name
    )


    copiar_verificado(
        origen,
        destino,
        sha,
        f"G1.11/{nombre}",
    )


    artefactos_locales_g111[
        nombre
    ] = str(
        destino
    )


# =============================================================================
# 25. RUNTIME
# =============================================================================

def version_paquete(
    nombre: str,
) -> str | None:

    try:

        return importlib.metadata.version(
            nombre
        )

    except importlib.metadata.PackageNotFoundError:

        return None


runtime = {
    "python": (
        sys.version.split()[0]
    ),

    "implementacion_python": (
        platform.python_implementation()
    ),

    "arquitectura": (
        platform.machine()
    ),

    "numpy": (
        np.__version__
    ),

    "pandas": (
        pd.__version__
    ),

    "pyarrow": (
        pa.__version__
    ),

    "duckdb": (
        duckdb.__version__
    ),

    "torch": (
        version_paquete(
            "torch"
        )
    ),

    "sentence_transformers": (
        version_paquete(
            "sentence-transformers"
        )
    ),
}


acelerador = {
    "torch_disponible": False,
    "cuda_disponible": False,
    "cuda_device": None,
}


try:

    import torch

    acelerador[
        "torch_disponible"
    ] = True


    acelerador[
        "cuda_disponible"
    ] = bool(
        torch.cuda.is_available()
    )


    if torch.cuda.is_available():

        acelerador[
            "cuda_device"
        ] = torch.cuda.get_device_name(
            0
        )


except Exception:

    pass


# =============================================================================
# 26. ESTADO LOCAL ÚNICO PARA NOTEBOOK 02
# =============================================================================

AHORA = datetime.now(
    timezone.utc
)

RUN_ID = AHORA.strftime(
    "%Y%m%dT%H%M%S_%fZ"
)


SECOP_RT_B02 = {
    "schema_version": "1.0.0",

    "bootstrap": (
        BOOTSTRAP
    ),

    "version_bootstrap": (
        VERSION_BOOTSTRAP
    ),

    "certificado": True,

    "run_id": (
        RUN_ID
    ),

    "timestamp_utc": (
        AHORA.isoformat()
    ),

    "punto_recuperacion": {
        "gate": (
            GATE_BASE_ESPERADO
        ),

        "manifiesto": str(
            ruta_g111
        ),

        "sha256_manifiesto": (
            SHA_MANIFIESTO_BASE_ESPERADO
        ),
    },

    "snapshot": {
        "snapshot_id": (
            SNAPSHOT_ID
        ),

        "fingerprint_parquet": (
            FINGERPRINT_PARQUET
        ),

        "filas": (
            FILAS_ARROW
        ),

        "columnas": (
            len(
                SCHEMA_LOCAL
            )
        ),

        "partes": (
            PARTES_ESPERADAS
        ),

        "ruta_parquet_local": str(
            RUTA_LOCAL_PARQUET
        ),
    },

    "referencia_unspsc": {
        "reference_id": (
            g111[
                "referencia_unspsc"
            ][
                "reference_id"
            ]
        ),

        "ruta_zip_local": str(
            ruta_zip_local
        ),

        "sha256_zip": (
            sha_zip_unspsc
        ),

        "ruta_catalogo_local": str(
            ruta_catalogo_local
        ),

        "sha256_catalogo": (
            sha_catalogo_unique
        ),
    },

    "g1_10": (
        artefactos_locales_g110
    ),

    "g1_11": (
        artefactos_locales_g111
    ),

    "runtime": (
        runtime
    ),

    "acelerador": (
        acelerador
    ),

    "siguiente_gate": (
        "G1.12-C"
    ),
}


escribir_json_atomico(
    RUTA_LOCAL_ESTADO,
    SECOP_RT_B02,
)


# =============================================================================
# 27. LOG PERSISTENTE DEL BOOTSTRAP
#
# NO sustituye ni modifica el checkpoint científico.
# =============================================================================

ruta_log_bootstrap = (
    RUTA_LOGS
    / (
        "bootstrap_02c_"
        f"{RUN_ID}.json"
    )
)


escribir_json_atomico(
    ruta_log_bootstrap,
    SECOP_RT_B02,
)


SHA_LOG_BOOTSTRAP = sha256_archivo(
    ruta_log_bootstrap
)


# =============================================================================
# 28. GARANTIZAR QUE EL CHECKPOINT GLOBAL NO FUE MODIFICADO
# =============================================================================

SHA_ESTADO_GLOBAL_DESPUES = sha256_archivo(
    RUTA_ESTADO_GLOBAL
)


if (
    SHA_ESTADO_GLOBAL_DESPUES
    != SHA_ESTADO_GLOBAL_ANTES
):

    raise RuntimeError(
        "BOOTSTRAP-02-C FAIL: el bootstrap modificó "
        "estado_laboratorio.json, lo cual está prohibido."
    )


# =============================================================================
# 29. CHECKS
# =============================================================================

checks = {
    "gate_base_g111": (
        estado.get(
            "ultimo_gate_cerrado"
        )
        == "G1.11-C"
    ),

    "sha_g111_exacto": (
        sha256_archivo(
            ruta_g111
        )
        == SHA_MANIFIESTO_BASE_ESPERADO
    ),

    "cadena_manifiestos_integra": (
        True
    ),

    "fingerprint_parquet_consistente": (
        g05[
            "parquet"
        ][
            "fingerprint_sha256"
        ]
        == FINGERPRINT_PARQUET
    ),

    "partes_188": (
        len(
            list(
                RUTA_LOCAL_PARQUET.glob(
                    "*.parquet"
                )
            )
        )
        == PARTES_ESPERADAS
    ),

    "filas_arrow": (
        FILAS_ARROW
        == FILAS_ESPERADAS
    ),

    "filas_duckdb": (
        FILAS_DUCKDB
        == FILAS_ESPERADAS
    ),

    "columnas_98": (
        len(
            SCHEMA_LOCAL
        )
        == COLUMNAS_ESPERADAS
    ),

    "parquet_todo_string": (
        all(
            pa.types.is_string(
                campo.type
            )
            for campo in SCHEMA_LOCAL
        )
    ),

    "zip_unspsc_integro": (
        sha256_archivo(
            ruta_zip_local
        )
        == sha_zip_unspsc
    ),

    "catalogo_unspsc_integro": (
        sha256_archivo(
            ruta_catalogo_local
        )
        == sha_catalogo_unique
    ),

    "checkpoint_global_no_modificado": (
        SHA_ESTADO_GLOBAL_ANTES
        == SHA_ESTADO_GLOBAL_DESPUES
    ),

    "estado_local_persistido": (
        RUTA_LOCAL_ESTADO.exists()
    ),

    "log_bootstrap_persistido": (
        ruta_log_bootstrap.exists()
    ),
}


if not all(
    checks.values()
):

    fallos = [
        nombre
        for nombre, aprobado
        in checks.items()
        if not aprobado
    ]


    raise RuntimeError(
        "BOOTSTRAP-02-C FAIL: "
        f"{fallos}"
    )


# =============================================================================
# 30. REPORTE
# =============================================================================

print()
print("=" * 108)
print("SECOP-RT LAB")
print("BOOTSTRAP-02-C — RESULTADO")
print("=" * 108)

print(
    f"Punto de recuperación:           "
    f"{GATE_BASE_ESPERADO}"
)

print(
    f"SHA manifiesto base:             "
    f"{SHA_MANIFIESTO_BASE_ESPERADO}"
)

print()
print("SNAPSHOT LOCAL")
print("-" * 108)

print(
    f"Snapshot:                        "
    f"{SNAPSHOT_ID}"
)

print(
    f"Partes verificadas:              "
    f"{PARTES_ESPERADAS}"
)

print(
    f"Partes copiadas:                 "
    f"{n_copiadas}"
)

print(
    f"Partes reutilizadas:             "
    f"{n_reutilizadas}"
)

print(
    f"Filas Arrow:                     "
    f"{FILAS_ARROW:,}"
)

print(
    f"Filas DuckDB:                    "
    f"{FILAS_DUCKDB:,}"
)

print(
    f"Columnas:                        "
    f"{len(SCHEMA_LOCAL)}"
)

print(
    f"Ruta local Parquet:              "
    f"{RUTA_LOCAL_PARQUET}"
)

print()
print("REFERENCIA UNSPSC")
print("-" * 108)

print(
    f"Reference ID:                    "
    f"{SECOP_RT_B02['referencia_unspsc']['reference_id']}"
)

print(
    f"ZIP local:                       "
    f"{ruta_zip_local}"
)

print(
    f"Catálogo local:                  "
    f"{ruta_catalogo_local}"
)

print()
print("RUNTIME")
print("-" * 108)

print(
    f"Python:                          "
    f"{runtime['python']}"
)

print(
    f"NumPy:                           "
    f"{runtime['numpy']}"
)

print(
    f"Pandas:                          "
    f"{runtime['pandas']}"
)

print(
    f"PyArrow:                         "
    f"{runtime['pyarrow']}"
)

print(
    f"DuckDB:                          "
    f"{runtime['duckdb']}"
)

print(
    f"PyTorch:                         "
    f"{runtime['torch']}"
)

print(
    f"sentence-transformers:           "
    f"{runtime['sentence_transformers']}"
)

print(
    f"CUDA disponible:                 "
    f"{acelerador['cuda_disponible']}"
)

print(
    f"CUDA device:                     "
    f"{acelerador['cuda_device']}"
)

print()
print("ESTADO")
print("-" * 108)

print(
    f"Estado local:                    "
    f"{RUTA_LOCAL_ESTADO}"
)

print(
    f"Log persistente:                 "
    f"{ruta_log_bootstrap}"
)

print(
    f"SHA log bootstrap:               "
    f"{SHA_LOG_BOOTSTRAP}"
)

print(
    "Checkpoint científico:           "
    "NO MODIFICADO"
)

print()
print("-" * 108)

for nombre, aprobado in checks.items():

    print(
        f"[{'PASS' if aprobado else 'FAIL'}] "
        f"{nombre}"
    )

print()
print("=" * 108)
print("RESULTADO BOOTSTRAP-02-C: PASS")
print("=" * 108)

print(
    "\nEl Notebook 02 quedó desacoplado del estado RAM "
    "del notebook anterior."
)

print(
    "Punto de continuación certificado: G1.12-C."
)

Mounted at /content/drive
SECOP-RT LAB
BOOTSTRAP-02-C — RECUPERACIÓN CERTIFICADA
Gate base:                       G1.11-C
Snapshot:                        secop2_jbjy-vk9h_ru1792151761_n5633184_20261016T181714_602776Z
Partes Parquet:                  188
Tamaño Parquet aprox.:           1.537 GiB

Reconstruyendo stage local...
  stage+SHA: 25/188
  stage+SHA: 50/188
  stage+SHA: 75/188
  stage+SHA: 100/188
  stage+SHA: 125/188
  stage+SHA: 150/188
  stage+SHA: 175/188
  stage+SHA: 188/188

SECOP-RT LAB
BOOTSTRAP-02-C — RESULTADO
Punto de recuperación:           G1.11-C
SHA manifiesto base:             9815f09a982d4caf299dfe7a9cca0f12d0173efa6db6c149cb44c04219a70e8e

SNAPSHOT LOCAL
------------------------------------------------------------------------------------------------------------
Snapshot:                        secop2_jbjy-vk9h_ru1792151761_n5633184_20261016T181714_602776Z
Partes verificadas:              188
Partes copiadas:                 188
Partes reutilizadas:           